# 적대적 AI — 전체 확장 실험 순차 실행

GPU 런타임을 선택하고 세 코드 셀을 순서대로 실행합니다. 준비된 비공개 Drive ZIP을 사용합니다. BIM·PGD 각 16조건, CNN 무방어 JSMA 781장, MobileNet PGD7 학습 3 epoch 및 원본/학습 모델 비교를 순차 실행합니다.

**실제 GPU 통합 실행은 미검증입니다.** CPU 중간 결과와 혼합하지 않고 별도 전체 실험을 생성합니다. 유료 GPU를 자동 구매하지 않습니다. Colab도 연결 종료나 이용 한도로 중단될 수 있으며, 저장된 완료 조건·표본·epoch만 복구 근거입니다. 이 노트북은 새 실행용이며 재실행 시 새 ID를 만듭니다. 중단 복구는 저장 결과 검사를 거쳐 별도로 진행합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys,os,json,zipfile,shutil,datetime,hashlib
ARCHIVE=Path('/content/drive/MyDrive/AdversarialAI_GPU_handoff_20261002.zip')
WORK=Path('/content/adversarial_full');WORK.mkdir(exist_ok=True)
REPO=WORK/'repo'
COMMIT='e974c4bb21ac2041dfbb93fdb18c1983dd6e402e'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/heechan9/AdversarialAI_Security.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',COMMIT],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==COMMIT
if not ARCHIVE.is_file():
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    sys.path.insert(0,str(REPO/'verification'))
    from colab_inputs import find_archive,download_archive
    credentials,_=google.auth.default()
    service=build('drive','v3',credentials=credentials,cache_discovery=False)
    ARCHIVE=download_archive(service,find_archive(service),WORK/'handoff.zip')
with zipfile.ZipFile(ARCHIVE) as z:
    selected=[i for i in z.infolist() if i.filename.startswith(('data/train-prepared/','data/valid/','data/test/','models/'))]
    for i in selected:
        target=(WORK/i.filename).resolve()
        assert target.is_relative_to(WORK.resolve()), 'unsafe archive path'
        assert (i.external_attr >> 16) & 0o170000 != 0o120000, 'symlink in archive'
        z.extract(i,WORK)
for p in (WORK/'models').glob('*.h5'):
    (REPO/'models').mkdir(exist_ok=True);shutil.copy2(p,REPO/'models'/p.name)
print('입력 압축 해제 완료. 학습 시작 전 코드가 이미지와 모델 해시를 검사합니다.')


In [ ]:
assert 'WORK' in globals() and 'REPO' in globals(), '첫 번째 셀을 먼저 오류 없이 완료하세요.'
VENV=WORK/'venv'
PY=str(VENV/'bin/python')
def run_logged(command,env=None):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env) as process:
        for line in process.stdout: print(line,end='',flush=True)
        if process.wait(): raise RuntimeError('위 설치/검사 로그를 확인하세요.')
# Colab system Python may omit ensurepip; host pip can populate a pip-free venv.
if not Path(PY).is_file():run_logged([sys.executable,'-m','venv','--without-pip',str(VENV)])
run_logged([sys.executable,'-m','pip','--python',PY,'install','tensorflow[and-cuda]==2.21.0','keras==3.15.1','numpy==2.3.5','Pillow','pandas','scikit-learn','matplotlib','pyyaml'])
ENV=os.environ.copy();ENV['PYTHONPATH']=str(REPO/'src')+':'+str(REPO)
ENV['TF_NUM_INTRAOP_THREADS']='2'
ENV['TF_NUM_INTEROP_THREADS']='1'
ENV['TF_FORCE_GPU_ALLOW_GROWTH']='true'
ENV['MPLBACKEND']='Agg'
CUDA_LIBS=sorted(str(p) for p in VENV.glob('lib/python*/site-packages/nvidia/*/lib'))
ENV['LD_LIBRARY_PATH']=':'.join(CUDA_LIBS+[ENV.get('LD_LIBRARY_PATH','')])
run_logged([PY,'-c',"import tensorflow as tf; g=tf.config.list_physical_devices('GPU'); print(g); assert g, 'GPU 미할당: CPU로 자동 진행하지 않습니다.'"],env=ENV)
DEST=Path('/content/drive/MyDrive/AdversarialAI_GPU_results');DEST.mkdir(exist_ok=True)
print('GPU 확인 완료. 다음 셀은 전체 5단계를 순차 실행합니다.')


In [ ]:
assert 'DEST' in globals() and 'PY' in globals(), '준비 셀을 먼저 실행하세요.'
import threading,time
RUN='full-gpu-'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S')
REMOTE=DEST/RUN
assert not REMOTE.exists(), '동일 실행 ID가 있습니다.'
REMOTE.mkdir()
STAGES=[]
for attack,steps,restarts,seed,batch in [('bim',10,1,0,32),('pgd',20,5,2026,16)]:
    out=REPO/'results/extensions/iterative'/(RUN+'-'+attack)
    command=[PY,'-u','-m','adversarial_ai.evaluation.iterative_evaluation','--attack',attack,'--steps',str(steps),'--step-size','.005','--restarts',str(restarts),'--seed',str(seed),'--batch-size',str(batch),'--data-dir',str(WORK/'data/test'),'--run-id',out.name]
    STAGES.append((attack,command,out,'run.json','COMPLETED_NOT_INDEPENDENT_APPROVAL','verification/iterative_result_audit.py'))
out=REPO/'results/extensions/jsma'/(RUN+'-jsma')
STAGES.append(('jsma',[PY,'-u','-m','adversarial_ai.evaluation.jsma_evaluation','--model','cnn','--defense','none','--theta','1','--gamma','.01','--max-steps','246','--limit','781','--data-dir',str(WORK/'data/test'),'--run-id',out.name],out,'run.json','FULL_COVERAGE_NOT_ROBUSTNESS_PROOF','verification/jsma_saved_audit.py'))
trained=REPO/'results/extensions/adversarial_training'/(RUN+'-mobilenet')
STAGES.append(('train',[PY,'-u','-m','adversarial_ai.training.adversarial_training','--model','mobilenet','--train-dir',str(WORK/'data/train-prepared/images'),'--validation-dir',str(WORK/'data/valid'),'--test-dir',str(WORK/'data/test'),'--epochs','3','--epsilon','.03','--step-size','.005','--steps','7','--seed','2026','--batch-size','8','--learning-rate','.00001','--run-id',trained.name],trained,'training.json','TRAINED_NOT_TEST_EVALUATED',None))
tested=REPO/'results/extensions'/('mobilenet-test-'+RUN)
STAGES.append(('test',[PY,'-u','verification/mobilenet_pgd7_test_evaluation.py','--trained-dir',str(trained),'--test-dir',str(WORK/'data/test'),'--output-dir',str(tested)],tested,'evaluation.json','COMPLETED',None))
# No CPU outputs are inherited: new GPU hardware requires a separate full series.
state={'source_commit':COMMIT,'run_id':RUN,'status':'STARTING','cpu_results_inherited':False,'completed':[],'commands':{stage:cmd for stage,cmd,*_ in STAGES}}
statefile=REMOTE/'pipeline.json'
def save_state():
    state['updated_at']=datetime.datetime.now(datetime.timezone.utc).isoformat()
    tmp=statefile.with_suffix('.tmp');tmp.write_text(json.dumps(state,indent=2)+'\n');tmp.replace(statefile)
def mirror(local):
    if not local.exists():return
    for p in sorted(local.rglob('*'),key=lambda p:(p.name=='state.json',str(p))):
        if p.is_symlink():raise ValueError('symlink in output')
        if not p.is_file() or p.suffix=='.tmp' or any(part.startswith('.pending-') for part in p.parts):continue
        dst=REMOTE/p.relative_to(REPO);dst.parent.mkdir(parents=True,exist_ok=True)
        sig=(p.stat().st_size,p.stat().st_mtime_ns)
        if copied.get(str(p))==sig and dst.exists():continue
        tmp=dst.with_name(dst.name+'.copying');shutil.copy2(p,tmp)
        if sig==(p.stat().st_size,p.stat().st_mtime_ns):tmp.replace(dst);copied[str(p)]=sig
        else:tmp.unlink(missing_ok=True)
copied={}
save_state()
try:
    for stage,command,out,report_name,expected,auditor in STAGES:
        assert not out.exists(), '기존 결과 덮어쓰기 금지: '+str(out)
        state['status']='RUNNING_'+stage;save_state()
        stop=threading.Event();errors=[]
        def periodic():
            while not stop.wait(30):
                try:mirror(out)
                except Exception as exc:errors.append(str(exc));return
        worker=threading.Thread(target=periodic,daemon=True);worker.start()
        process=None
        try:
            with (REMOTE/(stage+'.log')).open('x') as log:
                process=subprocess.Popen(command,cwd=REPO,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
                for line in process.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                    if errors:raise RuntimeError('Drive 백업 실패: '+errors[-1])
                if process.wait():raise RuntimeError(stage+' 실행 실패')
        finally:
            if process is not None and process.poll() is None:
                process.terminate()
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:process.kill();process.wait()
            stop.set();worker.join();mirror(out)
        if errors:raise RuntimeError('Drive 백업 실패: '+errors[-1])
        if report_name:
            report=json.loads((out/report_name).read_text())
            assert report['status']==expected, '완료 상태 불일치'
        if auditor:
            result=subprocess.run([PY,auditor,str(out)],cwd=REPO,env=ENV,capture_output=True,text=True,check=True)
            audit=json.loads(result.stdout)
            assert audit['result'] in ('FULL_OUTPUTS_CONSISTENT','FULL_SAVED_EVIDENCE_CONSISTENT')
            (REMOTE/(stage+'-audit.json')).write_text(json.dumps(audit,indent=2)+'\n')
        state['completed'].append(stage);save_state()
    state['status']='EXECUTED_AWAITING_FINAL_AUDIT_AND_MERGE';save_state()
except BaseException as exc:
    state['status']='INTERRUPTED' if isinstance(exc,KeyboardInterrupt) else 'ERROR'
    state['error']=str(exc);save_state();raise
print('전체 실행 종료. 최종 결과 감사·GitHub 반영·병합은 별도로 진행해야 합니다.')
print('결과 폴더:',REMOTE)


## 완료 판정
각 평가의 전체 표본 수와 저장 자료 감사를 확인합니다. 마지막 상태도 최종 승인이나 병합 완료를 뜻하지 않습니다. Drive 결과에는 비공개 학습 가중치가 있으므로 전체 폴더를 공개 저장소에 올리지 마세요.
